# Exercise 3 Digitization and Data Analytics - Machine Learning

## Supervised Learning - Decision Trees

Content

- Fundamental concepts in interaction with Apache Spark
- Usage of basic algorithms (Decision Trees) from the machine library MLLib within Spark

In [1]:
!pip install pyspark

  Using cached pyspark-4.1.2.tar.gz (455.4 MB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached py4j-0.10.9.9-py2.py3-none-any.whl.metadata (1.3 kB)
Using cached py4j-0.10.9.9-py2.py3-none-any.whl (203 kB)
  Created wheel for pyspark: filename=pyspark-4.1.2-py2.py3-none-any.whl size=456079515 sha256=f53cafd0054b17538760a2a7fc0acfaa1cd24d22b66fa30a8756c8435f084f07
  Stored in directory: /Users/krishnakumarm/Library/Caches/pip/wheels/e6/9c/35/b08622081a09dc48b9467b570ae170519430915aa3c8d27cf9
Successfully built pyspark
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pyspark]m1/2 [pyspark]


In [2]:
%matplotlib inline
import os


# Adapt the path to point to your iris.scale and iris.csv
scaledIrisDataPath = "iris.scale"
irisDataPath = "iris.csv"


import platform
print('the python version is: ' + platform.python_version())
import pyspark
from pyspark import SparkContext

ModuleNotFoundError: No module named 'matplotlib'

### Load the Decision Tree library:

In [ ]:
# $example on$
from pyspark.ml import Pipeline
from pyspark.ml.classification import DecisionTreeClassifier
from pyspark.ml.feature import StringIndexer, VectorIndexer, VectorAssembler
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
# $example off$
from pyspark.sql import SparkSession
from pyspark.sql.types import DoubleType, StructType, StructField, StringType

In [ ]:
spark = SparkSession.builder.appName("DecisionTreeClassificationExample").getOrCreate()

#### Exercise:
Load and investigate your data. What is the meaning of the values in column features? Are there any irregularities? 

In [ ]:
data = spark.read.format("libsvm").load(scaledIrisDataPath)
data.show(15, False)

#### Exercise:

1. Define a transformation, that adds indices to the labels in your data (use StringIndexer), and fit it to your data.
2. Define a transformation, that adds indices to the features in your data (use VectorIndexer), and fit it to your data.
3. Split your data into training and test data.
4. Show your test data, transformed with each of the defined transformation.

#### Deal with Categorical Label and Variables

In [ ]:
# Index labels, adding metadata to the label column.
# Fit on whole dataset to include all labels in index.
labelIndexer = StringIndexer(inputCol="label", outputCol="indexedLabel")
# Automatically identify categorical features, and index them.
# We specify maxCategories so features with > 4 distinct values are treated as continuous.
featureIndexer = VectorIndexer(inputCol="features", outputCol="indexedFeatures", maxCategories=4)


#### Split the data to training and test data sets

In [ ]:
# Split the data into training and test sets (30% held out for testing)
(trainingData, testData) = data.randomSplit([0.7, 0.3], seed=100)


#### Show your test data in a transformed manner

In [ ]:
#show indexed labels and features
labelIndexerModel = labelIndexer.fit(data)
featureIndexerModel = featureIndexer.fit(data)
trainingDataIndexed = labelIndexerModel.transform(trainingData)
trainingDataIndexed = featureIndexerModel.transform(trainingDataIndexed)
testDataIndexed = labelIndexerModel.transform(testData)
testDataIndexed = featureIndexerModel.transform(testDataIndexed)
testDataIndexed.select("label", "indexedLabel", "features", "indexedFeatures").show(15, False)


#### Fit Decision Tree Classification Model

Have a look at the documentation of the DecisionTreeClassifier() and its parameters 
(see https://spark.apache.org/docs/2.2.0/api/python/_modules/pyspark/ml/classification.html).

It is used to define the Decision Tree Part in the overall pipeline.

In [ ]:
# Train a DecisionTree model.
dt = DecisionTreeClassifier(labelCol="indexedLabel", featuresCol="indexedFeatures")

In [ ]:
help(DecisionTreeClassifier)

#### Pipeline Architecture

Define the overall pipeline. Be aware of the necessary transformations of your data before training a model.

In [ ]:
# Chain indexers and tree in a Pipeline
pipeline = Pipeline(stages=[labelIndexer, featureIndexer, dt])

# Train model.  This also runs the indexers.
model = pipeline.fit(trainingData)


#### Make predictions

In [ ]:
# Make predictions.
predictions = model.transform(testData)


#### Show both, the predictions and the labels in your data.

In [ ]:
# Select example rows to display.
predictions.select("prediction", "indexedLabel", "features").show(150)

In [ ]:
# Count mispredicted items.
predictions.filter(predictions.prediction != predictions.label).count()

#### Evaluation

In [ ]:
# Select (prediction, true label) and compute test error
evaluator = MulticlassClassificationEvaluator(labelCol="indexedLabel", predictionCol="prediction", metricName="accuracy")
accuracy = evaluator.evaluate(predictions)
print("Test Error = %g " % (1.0 - accuracy))

#### Exercise:
Change the partition of test data, rerun and compare the test errors.

#### Print out the generated model.

In [ ]:
treeModel = model.stages[2]
# summary only
print("Tree Model:")
print(treeModel)
# $example off$

In [ ]:
spark.stop()